# B.7 工具選擇怎樣才算可靠？


每題都選TOOL，在純計算試卷上也可能高分。混合DIRECT、TOOL、ASK題，才能量漏用、多用，以及該補問卻直接答。先用六筆人工結果讀這些分母：gold 是標準動作，predicted 是人工假設的選卡結果。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
gold = ["TOOL", "TOOL", "DIRECT", "DIRECT", "ASK", "ASK"]
predicted = ["TOOL", "DIRECT", "TOOL", "DIRECT", "ASK", "DIRECT"]
correct = sum(a == b for a, b in zip(gold, predicted, strict=True))
needed = sum(a == "TOOL" for a in gold)
missed = sum(a == "TOOL" and b != "TOOL" for a, b in zip(gold, predicted, strict=True))
not_needed = len(gold) - needed
extra = sum(a != "TOOL" and b == "TOOL" for a, b in zip(gold, predicted, strict=True))
print("動作選對", correct, "/", len(gold))
print("需要工具卻沒選", missed, "/", needed)
print("不需要工具卻選了", extra, "/", not_needed)

動作對3/6，兩題該用工具卻漏1，所以1/2；四題不該用工具卻多用1，所以1/4。後兩項分母不同，不能把都除六。選對卡也沒有實際運算，真呼叫與最後答案另計。

舊模型將同數字對的加法、解釋、照抄與工具狀態變化歸一個家族。相同問法留出新數字對得到96/96，但新問法只有42/48。總數還不錯，六道應用工具的題卻0/6，都是ASK：例如「幫我把1與2相加，給我答案」。這個錯例指出問法泛化不足，不是計算器返回錯數字。

另以完全相同問題只換system工具可用／停用：舊測試生成TOOL／ASK，支持有限模板內依狀態選擇。ASK仍只是一張卡，沒有輸出求助原因；資料裡人寫的「缺數量」不能當模型已解釋。

可靠性逐類、逐條件說明。不能把選卡器96/96與另一份JSON模型19/20拼成端到端成功率，也不能用42/48掩蓋新工具問法全部失敗。這些能力必須由同一模型、實際串接的完整軌跡支持。

<details>
<summary>補充：舊選卡器配方、題型與完整分項</summary>

正式選卡實驗把同一對數字的加法、照抄、解釋及工具狀態變化，放在同一個題目家族。家族先分成訓練、驗證與最後測試；同一筆問題不能在兩邊重複。訓練用來更新，驗證用來觀察，最後測試只在配方固定後核對，不按它的結果選步數或重訓。另一組未見問法則專門檢查：只換措辭，原本的選擇是否仍成立。因為題目來自有限模板，這些仍是受控的自然語言小世界，不能外推為任意問題的工具判斷能力。

數字範圍0至9，1加2與2加1同家族，1加1也保留；測的是未見數字組合，每個單獨數字仍可能在其他訓練家族出現。

每個家族都有以下四種任務；每個問法再配計算器可用、停用兩種system狀態。原問法每家族共有4種任務×2個問法×2種工具狀態＝16筆，未見問法診斷則是4×1×2＝8筆。

| 任務 | 原問法數／新問法數 | 工具可用的標準卡 | 工具停用的標準卡 |
| --- | --- | --- | --- |
| 求加法的精確數值 | 2／1 | TOOL | ASK |
| 用積木等方式解釋加法 | 2／1 | DIRECT | DIRECT |
| 照抄給出的兩個數字 | 2／1 | DIRECT | DIRECT |
| 報價，尚未提供購買數量 | 2／1 | ASK | ASK |

例如「單價 1 元，運費 2 元，總價多少？」在這份報價流程中還缺購買數量，標為ASK。六個家族在切分時放到最後檢查，含1與2的家族；每家族原問法16筆，共96題，新問法8筆，共48題。

選卡模型從隨機權重學示範答案，配方在最後檢查前固定。完整更新設定與原始生成保留在本節末的實測報告，沒有依最後結果重訓或改參數。

| 核對項目 | 同問法、未見數字對 | 未見問法診斷 |
| --- | --- | --- |
| 動作選對 | 96/96 | 42/48 |
| 應使用工具且選中TOOL | 12/12 | 0/6 |
| 不該使用工具卻選TOOL | 0/84 | 0/42 |
| 計算器停用卻選TOOL | 0/48 | 0/24 |
| 永遠選DIRECT的對照 | 48/96 | 24/48 |
| 永遠選TOOL的對照 | 12/96 | 6/48 |

表的第一列與最後兩個對照都以全部題數作分母。每個原問法家族有兩道「計算器可用的加法題」該選TOOL，六家族共12道；其餘96−12＝84道不該選。每家族16筆中有一半停用工具，所以停用分母為6×8＝48。診斷每家族只有一個新加法問法，因此相應分母變成6、48−6＝42與6×4＝24。這些子集有重疊，不能把各列分母相加當另一份考卷。

例如測試題「1 加 2 等於多少？」在system只寫「計算器可用。」時生成TOOL；把system換成「計算器停用。」就生成ASK。這輪訓練將工作策略放進示範答案，system只提供工具狀態，與[B.6](https://birdhackor.github.io/tiny-perceptron-vlm/B.6.html)為了說明資料欄位而人工寫出完整規則的短例不同。同一數字對的解釋與照抄題則生成DIRECT。這支持有限句型內已學到區分，不能證明模型先估算了自己算術答對的機率。

未見問法診斷沿用六個測試家族，但每種任務換一個新模板，各搭配可用、停用兩種狀態，共48題。計算器可用時，「幫我把 1 與 2 相加，給我答案。」卻生成ASK。六道應用工具的題全部如此；42/48的總分其實掩蓋了工具選擇在這個新問法上完全失敗。缺資訊的判定也遵守人工訂出的「報價先取得購買數量」規則，沒有測量模型是否能推導零單價等特例。

ASK只是本輪的一張卡，模型沒有產生「請補數量」或「請啟用工具」的完整解釋。資料另記求助原因，方便人查標註，不能把那些人寫的原因當成模型回答。

完整[實測報告](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/tool_choice.json)保留每題問題、工具狀態、標準動作與實際生成；[實驗程式](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/scripts/course_experiments/tool_choice.py)保留資料建立與切分方法。分數只評選卡，不含新JSON參數生成、工具運算或最後答案。即使選卡全對，也不能接用[B.4](https://birdhackor.github.io/tiny-perceptron-vlm/B.4.html)另一份模型的19/20成績，拼成一個沒有實際跑過的端到端成功率。

</details>
